# TP 2 — Qualité des données et jointures (santé)
**Durée indicative : 55–70 min**

Objectifs : lire des données synthétiques, repérer des anomalies, filtrer, dédupliquer et joindre deux tables.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


In [1]:
from pathlib import Path
import csv
from pyspark.sql import functions as F

# Databricks Runtime 14+ sets the current directory to the notebook's folder.
DATA_DIR = Path.cwd().parent / "data"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}. Open the notebook from the Git folder."

def read_small_csv(filename):
    # Python reads the small Git-tracked CSV on the driver; Spark does the processing.
    # For big data in production, read directly from cloud storage instead.
    with (DATA_DIR / filename).open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    return spark.createDataFrame(rows)


In [2]:
admissions_raw = read_small_csv("admissions_synthetiques.csv")
patients_raw = read_small_csv("patients_synthetiques.csv")
admissions_raw.show(5)
patients_raw.show(5)
admissions_raw.printSchema()

+------------+--------------+-------------+----------+-----------+-------+
|admission_id|date_admission|duree_minutes|patient_id|    service| statut|
+------------+--------------+-------------+----------+-----------+-------+
|        A001|    2026-10-02|          190|      P002|Cardiologie|termine|
|        A002|    2026-10-03|           40|      P003| Neurologie|termine|
|        A003|    2026-10-04|          190|      P004|  Pediatrie|termine|
|        A004|    2026-10-05|           40|      P005|   Urgences|termine|
|        A005|    2026-10-06|           90|      P006|Cardiologie|termine|
+------------+--------------+-------------+----------+-----------+-------+
only showing top 5 rows


+---+----------+----+
|age|patient_id|sexe|
+---+----------+----+
| 27|      P001|   F|
| 34|      P002|   F|
| 41|      P003|   F|
| 48|      P004|   F|
| 55|      P005|   F|
+---+----------+----+
only showing top 5 rows
root
 |-- admission_id: string (nullable = true)
 |-- date_admission: string (nullable = true)
 |-- duree_minutes: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- statut: string (nullable = true)



## 1. Construire des types cohérents
Les CSV contiennent du texte. Transformer les colonnes utiles en `date` et `int`.

In [3]:
admissions = (admissions_raw
    .withColumn("date_admission", F.to_date("date_admission"))
    .withColumn("duree_minutes", F.col("duree_minutes").cast("int")))
patients = patients_raw.withColumn("age", F.col("age").cast("int"))
admissions.printSchema()

root
 |-- admission_id: string (nullable = true)
 |-- date_admission: date (nullable = true)
 |-- duree_minutes: integer (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- statut: string (nullable = true)



**✅ Constat** — `date_admission` est maintenant de type `date` et `duree_minutes` de type `int` ; `age` est converti dans `patients`. Les identifiants (`admission_id`, `patient_id`) restent du texte : ce sont des codes, pas des quantités. En mode ANSI (par défaut sur Databricks et Spark 4), une valeur non convertible ferait échouer le cast ; sans ANSI, elle deviendrait `NULL`. Le rapport qualité ci-dessous vérifie donc aussi l'absence de `NULL` après conversion.

## 2. Contrôles de qualité
Repérer : identifiants d’admission dupliqués, services vides, durées négatives, patients absents du référentiel. **Les compter avant de supprimer.**

In [4]:
admissions.groupBy("admission_id").count().filter("count > 1").show()
admissions.filter((F.col("service") == "") | (F.col("duree_minutes") < 0)).show()
admissions.join(patients, "patient_id", "left_anti").show()

+------------+-----+
|admission_id|count|
+------------+-----+
|        A010|    2|
+------------+-----+



+------------+--------------+-------------+----------+--------+-------+
|admission_id|date_admission|duree_minutes|patient_id| service| statut|
+------------+--------------+-------------+----------+--------+-------+
|        A082|    2026-10-10|           90|      P004|        |termine|
|        A083|    2026-10-11|          -30|      P005|Urgences|termine|
+------------+--------------+-------------+----------+--------+-------+



+----------+------------+--------------+-------------+----------+-------+
|patient_id|admission_id|date_admission|duree_minutes|   service| statut|
+----------+------------+--------------+-------------+----------+-------+
|      P999|        A081|    2026-10-12|           45|Neurologie|termine|
+----------+------------+--------------+-------------+----------+-------+



In [5]:
# Complément : compter chaque anomalie AVANT de supprimer quoi que ce soit
nb_brut = admissions.count()
rapport_qualite = spark.createDataFrame([
    ("doublons exacts (lignes en trop)",          nb_brut - admissions.dropDuplicates().count()),
    ("admission_id présents plusieurs fois",      admissions.groupBy("admission_id").count().filter("count > 1").count()),
    ("service manquant (NULL ou vide)",           admissions.filter(F.col("service").isNull() | (F.col("service") == "")).count()),
    ("durée négative",                            admissions.filter(F.col("duree_minutes") < 0).count()),
    ("date ou durée NULL après conversion",       admissions.filter(F.col("date_admission").isNull() | F.col("duree_minutes").isNull()).count()),
    ("patient absent du référentiel",             admissions.join(patients, "patient_id", "left_anti").count()),
    ("patient_id en double dans le référentiel",  patients.groupBy("patient_id").count().filter("count > 1").count()),
], ["controle", "nb_lignes"])
print("Admissions brutes :", nb_brut, "| patients au référentiel :", patients.count())
rapport_qualite.show(truncate=False)

Admissions brutes : 84 | patients au référentiel : 31
+----------------------------------------+---------+
|controle                                |nb_lignes|
+----------------------------------------+---------+
|doublons exacts (lignes en trop)        |1        |
|admission_id présents plusieurs fois    |1        |
|service manquant (NULL ou vide)         |1        |
|durée négative                          |1        |
|date ou durée NULL après conversion     |0        |
|patient absent du référentiel           |1        |
|patient_id en double dans le référentiel|0        |
+----------------------------------------+---------+



**✅ Réponse — bilan qualité avant nettoyage**

Sur **84 admissions brutes** (et 31 patients au référentiel) :

| Anomalie | Nombre | Ligne concernée |
|---|---:|---|
| Doublon exact | 1 | `A010`, présente deux fois à l'identique |
| Service manquant | 1 | `A082` (champ vide) |
| Durée négative | 1 | `A083` (−30 min) |
| Patient inconnu | 1 | `A081` → `P999`, absent du référentiel |
| Valeur non convertible | 0 | — |
| Doublon dans le référentiel patients | 0 | — |

Il faut donc écarter **4 lignes** et retrouver **80 admissions** après nettoyage.

Remarques :
- le service manquant est une **chaîne vide** `""` et non un `NULL`, car `csv.DictReader` renvoie `""` pour un champ vide : d'où le double test `isNull()` **ou** `== ""` ;
- le patient `P031` existe dans le référentiel mais n'a aucune admission : ce n'est pas une anomalie ;
- l'unicité de `patient_id` dans le référentiel est indispensable : un patient en double **dupliquerait** ses admissions lors de la jointure.

## 3. Table fiabilisée
Supprimer doublons exacts, puis exclure les données impossibles / incomplètes ; ne garder que les admissions ayant un patient connu. Pour un pipeline réel, conserver les rejets dans une table dédiée.

In [6]:
admissions_clean = (admissions.dropDuplicates()
    .filter(F.col("service").isNotNull() & (F.col("service") != "") & (F.col("duree_minutes") >= 0))
    .join(patients.select("patient_id"), "patient_id", "inner"))
print("Brut :", admissions.count(), "| Nettoyé :", admissions_clean.count())

Brut : 84 | Nettoyé : 80


In [7]:
# Complément : conserver les rejets dans une table dédiée, avec leur motif
from pyspark.sql import Window

# Numéro de chaque ligne parmi ses copies exactes (1 = la première, 2 = un doublon)
rang_copie = F.row_number().over(Window.partitionBy(*admissions.columns).orderBy("admission_id"))

admissions_controlees = (admissions
    .withColumn("rang_copie", rang_copie)
    .join(patients.select("patient_id", F.lit(True).alias("patient_connu")), "patient_id", "left")
    .withColumn("motif_rejet",  # la première règle non respectée l'emporte
        F.when(F.col("rang_copie") > 1, "doublon exact")
         .when(F.col("service").isNull() | (F.col("service") == ""), "service manquant")
         .when(F.col("duree_minutes").isNull(), "durée manquante")
         .when(F.col("duree_minutes") < 0, "durée négative")
         .when(F.col("patient_connu").isNull(), "patient inconnu"))
    .drop("rang_copie", "patient_connu"))

rejets = admissions_controlees.filter(F.col("motif_rejet").isNotNull())
rejets.select("admission_id", "patient_id", "service", "duree_minutes", "motif_rejet").orderBy("admission_id").show()

# Réconciliation : aucune ligne ne disparaît sans explication
assert admissions.count() == admissions_clean.count() + rejets.count()
print(f"Brut {admissions.count()} = nettoyé {admissions_clean.count()} + rejets {rejets.count()}"
      f" (taux de rejet : {rejets.count() / admissions.count():.1%})")

+------------+----------+----------+-------------+----------------+
|admission_id|patient_id|   service|duree_minutes|     motif_rejet|
+------------+----------+----------+-------------+----------------+
|        A010|      P011|Neurologie|          240|   doublon exact|
|        A081|      P999|Neurologie|           45| patient inconnu|
|        A082|      P004|          |           90|service manquant|
|        A083|      P005|  Urgences|          -30|  durée négative|
+------------+----------+----------+-------------+----------------+



Brut 84 = nettoyé 80 + rejets 4 (taux de rejet : 4.8%)


**✅ Réponse — table fiabilisée**

On passe de **84 à 80 lignes**. Les 4 rejets sont exactement les anomalies comptées plus haut, chacune avec son motif : `A010` (doublon exact), `A081` (patient inconnu), `A082` (service manquant), `A083` (durée négative), soit un taux de rejet d'environ 4,8 %. La réconciliation *brut = nettoyé + rejets* (84 = 80 + 4) prouve qu'aucune ligne n'a disparu sans explication.

L'ordre des étapes compte : on déduplique d'abord, on applique ensuite les règles de validité, et on termine par la jointure `inner` avec le référentiel pour ne garder que les patients connus. Attention : le filtre `duree_minutes >= 0` écarte aussi les durées `NULL` sans le dire (une condition qui vaut `NULL` est traitée comme fausse par `filter`), d'où l'intérêt d'un motif explicite « durée manquante » dans la table des rejets.

## 4. Jointure
Récupérez l’âge et le sexe de chaque patient. Comparez `inner` et `left` sur le petit jeu de données.

In [8]:
visites_enrichies = admissions_clean.join(patients, "patient_id", "left")
visites_enrichies.select("admission_id", "service", "age", "duree_minutes").show(8)

+------------+-----------+---+-------------+
|admission_id|    service|age|duree_minutes|
+------------+-----------+---+-------------+
|        A071|  Pediatrie| 34|           40|
|        A041|Cardiologie| 34|           90|
|        A011|  Pediatrie| 34|          240|
|        A066| Neurologie| 69|          190|
|        A006| Neurologie| 69|          120|
|        A036|   Urgences| 69|           40|
|        A062| Neurologie| 41|           65|
|        A032|   Urgences| 41|          240|
+------------+-----------+---+-------------+
only showing top 8 rows


In [9]:
# Complément : comparer inner et left
print("Table nettoyée  -> inner :", admissions_clean.join(patients, "patient_id", "inner").count(),
      "| left :", admissions_clean.join(patients, "patient_id", "left").count())

# Sur les admissions brutes (dédupliquées), la différence apparaît
admissions_dedup = admissions.dropDuplicates()
print("Brut dédupliqué -> inner :", admissions_dedup.join(patients, "patient_id", "inner").count(),
      "| left :", admissions_dedup.join(patients, "patient_id", "left").count())

(admissions_dedup.join(patients, "patient_id", "left")
    .filter(F.col("age").isNull())
    .select("admission_id", "patient_id", "service", "age", "sexe")
    .show())

Table nettoyée  -> inner : 80 | left : 80


Brut dédupliqué -> inner : 82 | left : 83


+------------+----------+----------+----+----+
|admission_id|patient_id|   service| age|sexe|
+------------+----------+----------+----+----+
|        A081|      P999|Neurologie|NULL|NULL|
+------------+----------+----------+----+----+



**✅ Réponse — `inner` ou `left` ?**

- `inner` ne garde que les admissions dont le `patient_id` existe **des deux côtés** ;
- `left` garde **toutes** les admissions (la table de gauche) et met `NULL` dans `age` et `sexe` quand le patient est introuvable.

Sur la table nettoyée, aucune différence (**80 / 80**), puisque le nettoyage a déjà retiré le patient inconnu. Sur les admissions brutes dédupliquées : **82 en `inner` contre 83 en `left`**. La ligne en plus est `A081` (patient `P999`), avec un âge et un sexe à `NULL`. Un `inner` l'aurait fait disparaître **sans rien dire** ; le `left` (ou le `left_anti` du contrôle qualité) permet de la voir et de la compter.

## 5. Mini-défi
Quel service a le plus d’admissions ? Quelle est la durée moyenne par service ?

In [10]:
visites_enrichies.groupBy("service").agg(F.count("*").alias("admissions"), F.round(F.avg("duree_minutes"), 1).alias("duree_moyenne_min")).orderBy(F.desc("admissions")).show()

+-----------+----------+-----------------+
|    service|admissions|duree_moyenne_min|
+-----------+----------+-----------------+
|  Pediatrie|        20|            133.5|
|Cardiologie|        20|            118.8|
|   Urgences|        20|            129.3|
| Neurologie|        20|            106.5|
+-----------+----------+-----------------+



In [11]:
# Complément : classement reproductible (en cas d'égalité : durée moyenne, puis nom du service)
(visites_enrichies.groupBy("service")
    .agg(F.count("*").alias("admissions"),
         F.round(F.avg("duree_minutes"), 1).alias("duree_moyenne_min"))
    .orderBy(F.desc("admissions"), F.desc("duree_moyenne_min"), "service")
    .show())
print("Durée moyenne, tous services :", visites_enrichies.agg(F.round(F.avg("duree_minutes"), 1)).first()[0], "min")

+-----------+----------+-----------------+
|    service|admissions|duree_moyenne_min|
+-----------+----------+-----------------+
|  Pediatrie|        20|            133.5|
|   Urgences|        20|            129.3|
|Cardiologie|        20|            118.8|
| Neurologie|        20|            106.5|
+-----------+----------+-----------------+



Durée moyenne, tous services : 122.0 min


**✅ Réponse — mini-défi**

- **Aucun service n'a « le plus » d'admissions : c'est une égalité parfaite**, 20 admissions pour chacun des 4 services (80 au total). Piège : avec un tri sur `admissions` seul, l'ordre des ex æquo est arbitraire et peut changer d'une exécution à l'autre ; le premier affiché n'est pas « le gagnant ». Le second critère de tri ci-dessus rend le résultat reproductible.
- **Durée moyenne par service** : Pédiatrie **133,5 min**, Urgences **129,3 min**, Cardiologie **118,8 min**, Neurologie **106,5 min** (122,0 min tous services confondus). Les passages sont donc les plus longs en Pédiatrie et les plus courts en Neurologie.

### Discussion
Pourquoi vaut-il mieux compter et expliquer les rejets plutôt que supprimer silencieusement les lignes ? Pourquoi des données **réelles** de patients ne seraient-elles pas appropriées dans un dépôt GitHub public ?

**✅ Réponses de la discussion**

**Pourquoi compter et expliquer les rejets plutôt que supprimer en silence ?**
- **Traçabilité** : chaque chiffre se justifie (84 lignes reçues = 80 retenues + 4 rejetées, chacune avec un motif). Un indicateur sans cette réconciliation n'est pas auditable.
- **Mesure de la qualité** : le taux de rejet (environ 4,8 % ici) est un indicateur en soi. S'il grimpe d'un coup, c'est le signe d'un problème en amont (export cassé, format modifié…).
- **Correction à la source** : les rejets documentés sont renvoyés au système émetteur pour être corrigés, au lieu d'être perdus.
- **Éviter les biais** : si les rejets se concentrent sur un service ou une période, les indicateurs de ce service sont sous-estimés. Il faut le savoir pour interpréter les résultats.
- **Confiance** : à l'hôpital, des décisions (lits, personnel) reposent sur ces chiffres ; il faut pouvoir les rejouer et les expliquer.

**Pourquoi des données réelles de patients n'ont rien à faire dans un dépôt GitHub public ?**
- Les données de santé sont des données **sensibles** au sens du RGPD (article 9) : leur traitement est interdit sauf exceptions encadrées, avec une finalité précise, la minimisation des données et des mesures de sécurité.
- Elles relèvent du **secret médical**, et en France leur hébergement doit être confié à un hébergeur certifié **HDS** (Hébergeur de Données de Santé, article L.1111-8 du Code de la santé publique) : un dépôt public n'offre aucune de ces garanties.
- Un dépôt public est lisible par tout le monde, copié par les *forks*, les caches et les robots, et l'**historique Git** garde les fichiers même après leur suppression : une fuite est quasi irréversible.
- Retirer le nom ne suffit pas : la combinaison âge + sexe + date + service peut suffire à **ré-identifier** quelqu'un (pseudonymisé ne veut pas dire anonyme).
- Les conséquences : atteinte à la vie privée des patients, sanctions de la CNIL (jusqu'à 20 M€ ou 4 % du chiffre d'affaires mondial), responsabilité de l'établissement. D'où les données **synthétiques** de ce TP.